# The quantaq-cli Python library playbook

### This is an example workflow for QuantAQ users using the quantaq-cli Python library.

Let's import all functions in the toolkit module and configure the logging level.

In [1]:
from quantaq_cli import toolkit
from quantaq_cli.log import configure_logging

configure_logging("INFO")

We begin by concatenating together all raw files into a single DataFrame called concat_raw and saving it to concat_raw.csv in a temporary directory.

In [2]:
raw_files = [
    '../example_data/raw/MOD-00014-db-raw-file1.csv',
    '../example_data/raw/MOD-00014-db-raw-file2.csv'
]

concat_raw = toolkit.concat_files(raw_files)

concat_raw.to_csv("../example_data/tmp/concat_raw.csv", index=False)

2026-07-14 16:32:57.408 | INFO     | quantaq_cli.toolkit.concat:concat_files:19 - Parsing 2 files
2026-07-14 16:32:57.417 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.419 | ERROR    | quantaq_cli.schema:validate_schema:216 -   column  failure_case index   schema_context                                           check  check_number
0   None         False  None  DataFrameSchema  dataframe contains unstandardized column names             0
2026-07-14 16:32:57.419 | WARNING  | quantaq_cli.schema:validate_schema:222 - Standardizing unstandardized column names.
2026-07-14 16:32:57.420 | WARNING  | quantaq_cli.schema:validate_schema:226 - Coercing dtypes to expected types.
2026-07-14 16:32:57.426 | INFO     | quantaq_cli.schema:validate_schema:236 - Schema validation passed after coercion.
2026-07-14 16:32:57.431 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.432 | ERROR    | quantaq_cl

Let's do the same for all final files and call that concat_final.

In [3]:
final_files = [
    '../example_data/final/MOD-00014-db-final-file1.csv',
    '../example_data/final/MOD-00014-db-final-file2.csv'   
]

concat_final = toolkit.concat_files(final_files)

concat_final.to_csv("../example_data/tmp/concat_final.csv", index=False)


2026-07-14 16:32:57.449 | INFO     | quantaq_cli.toolkit.concat:concat_files:19 - Parsing 2 files
2026-07-14 16:32:57.454 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.454 | ERROR    | quantaq_cli.schema:validate_schema:216 -   column  failure_case index   schema_context                                           check  check_number
0   None         False  None  DataFrameSchema  dataframe contains unstandardized column names             0
2026-07-14 16:32:57.455 | WARNING  | quantaq_cli.schema:validate_schema:222 - Standardizing unstandardized column names.
2026-07-14 16:32:57.456 | WARNING  | quantaq_cli.schema:validate_schema:226 - Coercing dtypes to expected types.
2026-07-14 16:32:57.459 | INFO     | quantaq_cli.schema:validate_schema:236 - Schema validation passed after coercion.
2026-07-14 16:32:57.463 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.463 | ERROR    | quantaq_cl

At this point, we have two large files. Next, we will ``merge`` the two files together
into a single DataFrame called merged:

In [4]:
tmp_files = [
    "../example_data/tmp/concat_raw.csv",
    "../example_data/tmp/concat_final.csv"
]

merged = toolkit.merge_files(tmp_files)

2026-07-14 16:32:57.475 | INFO     | quantaq_cli.toolkit.merge:merge_files:73 - Parsing 2 files


Next's let's re-flag the data to make sure it matches QuantAQ's QA/QC checks.

In [5]:
flagged = toolkit.flag_dataframe(merged)

2026-07-14 16:32:57.502 | INFO     | quantaq_cli.utilities:infer_data_source:50 - Reading mostly 60.0s data --> inferring database
2026-07-14 16:32:57.504 | INFO     | quantaq_cli.toolkit.flag:add_flag:123 - Flagged: FLAG_CO (flag 16) --> 60 rows


We can check the flag summary using echo_flag_table:

In [6]:
toolkit.echo_flag_table(flagged)

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ FLAG          ┃ FLAG VALUE ┃ # OCCURENCES ┃ % DATA ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━┩
│ FLAG_STARTUP  │          1 │            0 │    0.0 │
│ FLAG_OPC      │          2 │            0 │    0.0 │
│ FLAG_NEPH     │          4 │            0 │    0.0 │
│ FLAG_RHT      │          8 │           60 │  100.0 │
│ FLAG_CO       │         16 │           60 │  100.0 │
│ FLAG_NO       │         32 │            0 │    0.0 │
│ FLAG_NO2      │         64 │            0 │    0.0 │
│ FLAG_O3       │        128 │            0 │    0.0 │
│ FLAG_CO2      │        256 │            0 │    0.0 │
│ FLAG_SO2      │        512 │            0 │    0.0 │
│ FLAG_H2S      │       1024 │            0 │    0.0 │
│ FLAG_BAT      │       2048 │            0 │    0.0 │
│ FLAG_OVERHEAT │       4096 │            0 │    0.0 │
└───────────────┴────────────┴──────────────┴────────┘

Now let's expunge the data, which means we will NaN the corresponding flagged columns.

In [7]:
expunged = toolkit.expunge_dataframe(flagged)

At this point we have 1-minute data, which is a lot of data! Let's resample to a 5min frequency:

In [8]:
five_min = toolkit.resample_dataframe(expunged, '5min')

In [9]:
tmp_files = [
    "../tests/files/modulair/MOD-00014-db-raw.csv",
    "../tests/files/modulair/MOD-00014-db-final.csv"
]

merged = toolkit.merge_files(tmp_files)


2026-07-14 16:32:57.563 | INFO     | quantaq_cli.toolkit.merge:merge_files:73 - Parsing 2 files
2026-07-14 16:32:57.568 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.569 | ERROR    | quantaq_cli.schema:validate_schema:216 -   column  failure_case index   schema_context                                           check  check_number
0   None         False  None  DataFrameSchema  dataframe contains unstandardized column names             0
2026-07-14 16:32:57.569 | WARNING  | quantaq_cli.schema:validate_schema:222 - Standardizing unstandardized column names.
2026-07-14 16:32:57.570 | WARNING  | quantaq_cli.schema:validate_schema:226 - Coercing dtypes to expected types.
2026-07-14 16:32:57.576 | INFO     | quantaq_cli.schema:validate_schema:236 - Schema validation passed after coercion.
2026-07-14 16:32:57.583 | ERROR    | quantaq_cli.schema:validate_schema:215 - Schema validation failed.
2026-07-14 16:32:57.584 | ERROR    | quantaq_cli.

And that's it!